In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.cluster import KMeans

## EDA

In [157]:
dataset = pd.read_csv("../datasets/imdb_top_1000.csv")
dataset.iloc[1:5, :]

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,"57,300,000"
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,"4,360,000"


In [158]:
dataset.isnull().sum()

Poster_Link        0
Series_Title       0
Released_Year      0
Certificate      101
Runtime            0
Genre              0
IMDB_Rating        0
Overview           0
Meta_score       157
Director           0
Star1              0
Star2              0
Star3              0
Star4              0
No_of_Votes        0
Gross            169
dtype: int64

### String Cleaning

In [159]:
dataset['Runtime'] = dataset['Runtime'].str.removesuffix(' min').astype('int64')
dataset['Gross'] = dataset['Gross'].str.replace(',', '')
dataset.head()

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,28341469
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,134966411
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,534858444
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,57300000
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,4360000


### Fixing Gross

In [160]:
print(f"NaN in 'Gross' before: {np.size(dataset[dataset['Gross'].isna()])}")
dataset['Gross'] = dataset['Gross'].replace(np.nan, 0)
print(f"NaN in 'Gross' after: {np.size(dataset[dataset['Gross'].isna()])}")
dataset['Gross'] = dataset['Gross'].astype('int64')

NaN in 'Gross' before: 2704
NaN in 'Gross' after: 0


### Dict of Genres

In [161]:
genre = defaultdict(list)
for idx, genres in enumerate(dataset['Genre']):
    for y in genres.split(','):
        genre[y.strip()].append(dataset.iloc[idx]['Series_Title'])

genre

defaultdict(list,
            {'Drama': ['The Shawshank Redemption',
              'The Godfather',
              'The Dark Knight',
              'The Godfather: Part II',
              '12 Angry Men',
              'The Lord of the Rings: The Return of the King',
              'Pulp Fiction',
              "Schindler's List",
              'Fight Club',
              'The Lord of the Rings: The Fellowship of the Ring',
              'Forrest Gump',
              'The Lord of the Rings: The Two Towers',
              'Goodfellas',
              "One Flew Over the Cuckoo's Nest",
              'Hamilton',
              'Gisaengchung',
              'Soorarai Pottru',
              'Interstellar',
              'Cidade de Deus',
              'Saving Private Ryan',
              'The Green Mile',
              'La vita è bella',
              'Se7en',
              'The Silence of the Lambs',
              'Seppuku',
              'Shichinin no samurai',
              "It's a Wonderful 

### Genre counts

In [162]:
for g in set(genre.keys()):
    dataset[g] = dataset['Genre'].apply(lambda x: 1 if g in x.strip() else 0)

In [172]:
dataset.loc[:, list(genre.keys())].sum()

Drama        724
Crime        209
Action       189
Adventure    196
Biography    109
History       56
Sci-Fi        67
Romance      125
Western       20
Fantasy       66
Comedy       233
Thriller     137
Animation     82
Family        56
War           51
Mystery       99
Music         47
Horror        32
Musical       17
Film-Noir     19
Sport         19
dtype: int64

### Weighted Avg to Supplement Missing Meta Scores

In [173]:
ratings = dataset['IMDB_Rating'].values
votes = dataset['No_of_Votes'].values
dataset['Weighted_Avg'] = (ratings * votes + votes.min() * ratings.mean()) / (votes + votes.min())
dataset.head()

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,...,Crime,Music,War,Musical,Horror,Fantasy,Action,Adventure,Biography,Weighted_Avg
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,...,0,0,0,0,0,0,0,0,0,9.285691
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,...,1,0,0,0,0,0,0,0,0,9.180931
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,...,1,0,0,0,0,0,1,0,0,8.988679
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,...,1,0,0,0,0,0,0,0,0,8.977178
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,...,1,0,0,0,0,0,0,0,0,8.963129


### Dataset describe

In [174]:
dataset.describe()

,Runtime,IMDB_Rating,Meta_score,No_of_Votes,Gross,Drama,Western,Comedy,Romance,Sport,...,Crime,Music,War,Musical,Horror,Fantasy,Action,Adventure,Biography,Weighted_Avg
count,1000.000000,1000.000000,843.000000,1.000000e+03,1.000000e+03,1000.00000,1000.00000,1000.000000,1000.000000,1000.000000,...,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,122.891000,7.949300,77.971530,2.736929e+05,5.653688e+07,0.72400,0.02000,0.233000,0.125000,0.019000,...,0.209000,0.047000,0.051000,0.017000,0.032000,0.066000,0.189000,0.196000,0.109000,7.957808
std,28.093671,0.275491,12.376099,3.273727e+05,1.032382e+08,0.44724,0.14007,0.422954,0.330884,0.136593,...,0.406798,0.211745,0.220108,0.129336,0.176088,0.248406,0.391705,0.397167,0.311795,0.242824
min,45.000000,7.600000,28.000000,2.508800e+04,0.000000e+00,0.00000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.611562
25%,103.000000,7.700000,70.000000,5.552625e+04,4.457098e+05,0.00000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.799293
50%,119.000000,7.900000,79.000000,1.385485e+05,1.070275e+07,1.00000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.913042
75%,137.000000,8.100000,87.000000,3.741612e+05,6.153989e+07,1.00000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.077983
max,321.000000,9.300000,100.000000,2.343110e+06,9.366622e+08,1.00000,1.00000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,9.285691


## ML stuff

In [ ]:
height = dataset.shape[0]
train_X = dataset.iloc[:int(.8*height)]

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,...,Crime,Music,War,Musical,Horror,Fantasy,Action,Adventure,Biography,Weighted_Avg
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,...,0,0,0,0,0,0,0,0,0,9.285691
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,...,1,0,0,0,0,0,0,0,0,9.180931
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,...,1,0,0,0,0,0,1,0,0,8.988679
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,...,1,0,0,0,0,0,0,0,0,8.977178
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,...,1,0,0,0,0,0,0,0,0,8.963129
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,https://m.media-amazon.com/images/M/MV5BYzVmYz...,Ocean's Eleven,2001,UA,116,"Crime, Thriller",7.7,Danny Ocean and his ten accomplices plan to ro...,74.0,Steven Soderbergh,...,1,0,0,0,0,0,0,0,0,7.711551
796,https://m.media-amazon.com/images/M/MV5BNTIyNT...,Vampire Hunter D: Bloodlust,2000,U,103,"Animation, Action, Fantasy",7.7,"When a girl is abducted by a vampire, a legend...",62.0,Yoshiaki Kawajiri,...,0,0,0,0,0,1,1,0,0,7.815187
797,https://m.media-amazon.com/images/M/MV5BMjZkOT...,"O Brother, Where Art Thou?",2000,U,107,"Adventure, Comedy, Crime",7.7,"In the deep south during the 1930s, three esca...",69.0,Joel Coen,...,1,0,0,0,0,0,0,1,0,7.720057
798,https://m.media-amazon.com/images/M/MV5BZDYwYz...,Interstate 60: Episodes of the Road,2002,R,116,"Adventure, Comedy, Drama",7.7,"Neal Oliver, a very confused young man and an ...",NaN,Bob Gale,...,0,0,0,0,0,0,0,1,0,7.813537
